# Day 2 · Lab 2 (Part 2) — Tree ensembles and cautious model comparison

## Purpose
This lab builds on the single Decision Tree from Part 1 (Decision Trees).

The two **core** methods are **Random Forest** (bagging) and **Gradient Boosting** (boosting). These are the required ensemble methods for Day 2: together they cover the main teaching goals for structured engineering tabular data — ensemble intuition, tuning, diagnostics, and practical comparison. We compare them against the Day 1 Ridge baseline and the Day 2 single-tree reference.

**XGBoost** and **LightGBM** are included only as an **optional extension** (Section 11), to be run after the core Random Forest / Gradient Boosting work if time and the environment allow. AdaBoost is intentionally left out of the core teaching path: it adds little pedagogically here beyond what Gradient Boosting already shows.

## Learning goals
By the end of this lab, you should be able to:

- explain why ensembles often improve on a single tree;
- train a `RandomForestRegressor` (bagging);
- train a histogram-based Gradient Boosting model (boosting) and see how it improves progressively;
- understand where XGBoost and LightGBM fit in the boosting family (optional extension);
- compare models using validation and test metrics;
- use cross-validation as an internal check without replacing the organiser-defined test set;
- inspect regime-wise errors;
- compute permutation importance with appropriate caution;
- distinguish regime-shift from source-shift out-of-domain data;
- state one safe-use limitation for deployment.

## Expected outputs
By the end of the lab, you should have:

1. Ridge, Decision Tree, Random Forest, and Gradient Boosting results;
2. a small cross-validation comparison;
3. a view of Gradient Boosting improving iteration by iteration;
4. optional XGBoost and LightGBM comparisons if those libraries are installed;
5. parity and residual plots;
6. regime-wise error tables;
7. permutation importance;
8. out-of-domain (regime-shift vs source-shift) diagnostics;
9. a short engineering recommendation.

> Important: The best validation metric is not the whole decision. For engineering use, also ask where the model fails, whether the data are in-domain, and whether the explanation is stable enough to communicate.

## 1. Imports and environment check

Run this cell first. It checks the core packages needed for Day 2 Lab 2 (Part 2).

In [ ]:
from pathlib import Path
import sys
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree

print("Python executable :", sys.executable)
print("Python version    :", platform.python_version())
print("Platform          :", platform.platform())
print("\nCore imports succeeded.")

# Core ensemble methods for this lab: Random Forest (bagging) and
# histogram-based Gradient Boosting (boosting).
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate

Python executable : /opt/homebrew/Caskroom/miniforge/base/envs/ai4nth/bin/python
Python version    : 3.11.15
Platform          : macOS-26.5.1-arm64-arm-64bit

Core imports succeeded.


## 2. Locate the dataset folder

Shared datasets are stored in the repository-root `datasets/` folder. This notebook is located under `course_materials/day2_tree_models/notebooks/`, so the code below locates the repository root and then uses `datasets/` for the CHF train, validation, and test files.


In [ ]:
notebook_dir = Path.cwd().resolve()
bundle_root = notebook_dir.parents[2]
data_dir = bundle_root / "datasets"

print("Notebook directory :", notebook_dir)
print("Bundle root        :", bundle_root)
print("Dataset directory  :", data_dir)

assert data_dir.exists(), f"Dataset directory not found: {data_dir}"
print("\nPASS: dataset directory exists.")

## 3. Load the train / validation / test data

The main comparison uses the organiser-defined split.

In [ ]:
# =========================
# EDIT THESE FILENAMES ONLY IF YOUR DATA FOLDER IS DIFFERENT
# =========================

train_file = "chf_train.csv"
val_file = "chf_val.csv"
test_file = "chf_test.csv"

ood_files = [
    "chf_OOD_Kim2000.csv",
    "chf_OOD_Lee1966.csv",
    "chf_OOD_Peterlongo1966.csv",
]

target_col = "CHF (kW/m^2)"
id_col = "Number"
reference_col = "Reference name"


def load_csv(data_dir: Path, filename: str) -> pd.DataFrame:
    path = data_dir / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path)


train_df = load_csv(data_dir, train_file)
val_df = load_csv(data_dir, val_file)
test_df = load_csv(data_dir, test_file)

print(f"Train shape: {train_df.shape}")
print(f"Val shape  : {val_df.shape}")
print(f"Test shape : {test_df.shape}")

## 4. Quick inspection of the data

Confirm the shape, target column, source distribution, and missing values before modelling.

In [ ]:
assert target_col in train_df.columns, f"Target column '{target_col}' not found in training data."

print("Columns in the dataset:")
for c in train_df.columns:
    print("-", c)

print("\nTraining data preview:")
display(train_df.head())

print("\nRows by reference:")
display(train_df[reference_col].value_counts().to_frame("train rows").head(12))

print("\nMissing values by split:")
missing_summary = pd.DataFrame({
    "train_missing": train_df.isna().sum(),
    "val_missing": val_df.isna().sum(),
    "test_missing": test_df.isna().sum(),
})
display(missing_summary)

## 5. Define features and target

Start with Day 1's three-feature setup for direct comparison. For an extension, set `use_all_physical_features = True` and rerun the notebook.

In [ ]:
day1_feature_cols = [
    "Pressure (kPa)",
    "Mass Flux (kg/m^2/s)",
    "Outlet Quality",
]

all_physical_feature_cols = [
    "Tube Diameter (m)",
    "Heated Length (m)",
    "Pressure (kPa)",
    "Mass Flux (kg/m^2/s)",
    "Outlet Quality",
    "Inlet Subcooling (kJ/kg)",
    "Inlet Temperature (degreeC )",
]

# Start with the Day 1 features so the comparison is direct.
# Change this to True during the extension if you want to use all physical inputs.
use_all_physical_features = False

feature_cols = all_physical_feature_cols if use_all_physical_features else day1_feature_cols

print("Selected feature columns:")
for c in feature_cols:
    print("-", c)

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

print("\nShapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape, "y_val  :", y_val.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape)

## 6. Helper functions

These keep **preprocessing, metrics, and display** consistent across every model, so the comparison stays fair. Same setup as Part 1 — a quick reminder of the three scores each table reports:

- **MAE** (kW/m²): the average size of the error.
- **RMSE** (kW/m²): like MAE but penalises large misses more; it is the score we **rank models by**.
- **R²** (≤ 1): the fraction of variance explained. 1 = perfect, 0 = no better than guessing the mean, **negative = worse than the mean** (you will see this on out-of-domain data).

As before, only the **linear Ridge** model is scaled (`scale_numeric=True`); trees and tree ensembles are scale-invariant, so they use `scale_numeric=False`.


In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


def make_preprocessor(X, scale_numeric: bool = False):
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    transformers = [("num", Pipeline(steps=numeric_steps), numeric_cols)]

    if categorical_cols:
        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
        ])
        transformers.append(("cat", categorical_transformer, categorical_cols))

    return ColumnTransformer(transformers=transformers, remainder="drop")


def evaluate_on_splits(model_name, model, include_train=False):
    split_data = []
    if include_train:
        split_data.append(("Train", X_train, y_train))
    split_data.extend([
        ("Validation", X_val, y_val),
        ("Test", X_test, y_test),
    ])

    rows = []
    predictions = {}
    for split_name, X, y in split_data:
        pred = model.predict(X)
        row = regression_metrics(y, pred)
        row["Model"] = model_name
        row["Split"] = split_name
        rows.append(row)
        predictions[split_name] = pred

    return pd.DataFrame(rows), predictions


def display_metric_table(results_df):
    ordered = results_df[["Model", "Split", "MAE", "RMSE", "R2"]].copy()
    display(ordered.style.format({"MAE": "{:.2f}", "RMSE": "{:.2f}", "R2": "{:.3f}"}))

## 7. Ensemble model map

A single tree is **unstable**: small changes in the data can flip its splits, and a deep tree overfits. Ensembles fix this by combining **many** trees so their individual errors partly cancel. Two different strategies:

- **Bagging** (e.g. **Random Forest**): train many trees **in parallel**, each on a bootstrap resample of the rows and a random subset of features, then **average** them. Averaging cancels the random part of each tree's error, so it mainly reduces **variance** (overfitting). Each tree is deep and overfits on its own, but their average is stable.
- **Boosting** (e.g. **Gradient Boosting**): train trees **sequentially**, each new (usually shallow) tree fitting the **residual error** left by the ones before it. This mainly reduces **bias** (underfitting), assembling one strong model from many weak ones.
- **Stacking**: train several different models, then train a second-level model to learn how best to combine them.

For this lab the two **core** methods are Random Forest (bagging) and Gradient Boosting (boosting) — the two most common, best-understood choices for structured engineering tables. AdaBoost is skipped in the core path; XGBoost / LightGBM appear later as an optional extension.

> One-line intuition: **bagging = average many over-fit trees to kill variance; boosting = add many weak trees to kill bias.**


## 8. Define the candidate models

Four models: two reference baselines (Ridge, single Decision Tree) plus the two **core** ensembles (Random Forest, Histogram Gradient Boosting). The settings below are deliberately modest so the lab runs on a standard laptop.

**The knobs worth noticing** (feel free to tune them later):

- **Random Forest** — `n_estimators` (how many trees to average; more = smoother but slower), plus per-tree limits like `max_depth` / `min_samples_leaf`. RF trees are allowed to grow fairly deep because *averaging* controls their variance.
- **Gradient Boosting** — `max_iter` (how many sequential trees), `learning_rate` (how big a correction each tree makes; smaller is more accurate but needs more trees), and `max_leaf_nodes` (per-tree complexity). `learning_rate` and `max_iter` trade off against each other.

> Note on the single tree: the `Decision Tree` here uses fixed settings (`max_depth=6, min_samples_leaf=30`) as a **stable reference point**, not the CV-selected tree from Part 1 (Decision Trees). We want a fixed single-tree baseline to judge how much the ensembles add, so do not expect its numbers to match Part 1 exactly.


In [ ]:
models = {
    "Ridge": Pipeline(steps=[
        ("preprocess", make_preprocessor(X_train, scale_numeric=True)),
        ("model", Ridge(alpha=1.0)),
    ]),
    "Decision Tree": Pipeline(steps=[
        ("preprocess", make_preprocessor(X_train, scale_numeric=False)),
        ("model", DecisionTreeRegressor(
            max_depth=6,
            min_samples_leaf=30,
            random_state=42,
        )),
    ]),
    "Random Forest": Pipeline(steps=[
        ("preprocess", make_preprocessor(X_train, scale_numeric=False)),
        ("model", RandomForestRegressor(
            n_estimators=120,
            max_depth=14,
            min_samples_leaf=5,
            max_features=1.0,
            random_state=42,
            n_jobs=1,
        )),
    ]),
    "Histogram Gradient Boosting": Pipeline(steps=[
        ("preprocess", make_preprocessor(X_train, scale_numeric=False)),
        ("model", HistGradientBoostingRegressor(
            max_iter=180,
            learning_rate=0.06,
            max_leaf_nodes=31,
            min_samples_leaf=25,
            l2_regularization=0.0,
            random_state=42,
        )),
    ]),
}

# Ridge and Decision Tree are reference baselines (linear + single tree).
# Random Forest and Histogram Gradient Boosting are the two core ensemble
# methods this lab focuses on.
print("Candidate models:")
for name in models:
    print("-", name)

## 9. Small internal cross-validation check

Before fitting on all the data, we run a quick **3-fold cross-validation on the training set** to gauge how stable each model is. (Same idea as Part 1's CV: train on some folds, score on the held-out fold, rotate.) It does **not** replace the organiser-defined validation and test sets — it is only an early stability check.

**What to look for:**

- **Lower `CV RMSE mean`** = better average accuracy. Expect Random Forest and Gradient Boosting to sit below the single tree and Ridge.
- **Smaller `CV RMSE std`** = more stable across folds (less sensitive to which rows it trained on). Ensembles are usually more stable than a single tree.


In [ ]:
cv = KFold(n_splits=3, shuffle=True, random_state=42)
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
}

cv_rows = []
for model_name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )
    cv_rows.append({
        "Model": model_name,
        "CV RMSE mean": -scores["test_rmse"].mean(),
        "CV RMSE std": scores["test_rmse"].std(),
        "CV MAE mean": -scores["test_mae"].mean(),
        "CV R2 mean": scores["test_r2"].mean(),
    })

cv_results = pd.DataFrame(cv_rows).sort_values("CV RMSE mean")
display(cv_results.style.format({
    "CV RMSE mean": "{:.2f}",
    "CV RMSE std": "{:.2f}",
    "CV MAE mean": "{:.2f}",
    "CV R2 mean": "{:.3f}",
}))

## 10. Fit each model on the full training split

After the CV check, we fit each model once on **all** training rows and score it on train, validation, and test.

**How to read the comparison table:**

- Compare models on **Validation / Test RMSE** — the ensembles (Random Forest, Gradient Boosting) should beat both the single Decision Tree and Ridge. That gap is the payoff of ensembling.
- Look at each model's **Train vs Validation gap**. A modest gap is healthy. Note that **Random Forest often has a very low Train RMSE yet still generalises well** — that large train-vs-validation gap is *not* the same failure as the single deep tree in Part 1, because averaging many trees controls the variance. Always judge a model by its **held-out** score, not its train score.
- The "Best validation RMSE" line names the front-runner, but keep reading: the sections below check *where* it wins and where it might still fail.


In [ ]:
fitted_models = {}
all_results = []
all_predictions = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[model_name] = model
    results, predictions = evaluate_on_splits(model_name, model, include_train=True)
    all_results.append(results)
    all_predictions[model_name] = predictions

comparison = pd.concat(all_results, ignore_index=True)
display_metric_table(comparison)

print("Best validation RMSE:")
display(
    comparison[comparison["Split"] == "Validation"]
    .sort_values("RMSE")
    .head(1)
    .style.format({"MAE": "{:.2f}", "RMSE": "{:.2f}", "R2": "{:.3f}"})
)

## 10b. Watching boosting learn, one tree at a time

Lecture 2.2 described boosting as **progressive**: unlike a Random Forest (many independent trees averaged at once), a boosting model adds trees **sequentially**, each new tree fitting the error left by the ones before it.

We can make that visible. `staged_predict` replays the Histogram Gradient Boosting model after every added tree, so we can plot how train and validation RMSE evolve as the ensemble grows. Two teaching points to look for:

- **Diminishing returns.** RMSE drops fast at first, then flattens — the first trees do most of the work.
- **Where overfitting would begin.** If validation RMSE turns back up while training RMSE keeps falling, the extra trees are memorising noise. That crossover point is exactly what early stopping / `max_iter` tuning protects against.

In [ ]:
# Boosting adds trees one at a time, each correcting the running error.
# staged_predict lets us watch RMSE fall as trees accumulate.
boost_name = "Histogram Gradient Boosting"
boost_pipe = fitted_models[boost_name]
boost_pre = boost_pipe.named_steps["preprocess"]
boost_model = boost_pipe.named_steps["model"]

X_train_t = boost_pre.transform(X_train)
X_val_t = boost_pre.transform(X_val)

train_rmse_stages = [
    root_mean_squared_error(y_train, pred) for pred in boost_model.staged_predict(X_train_t)
]
val_rmse_stages = [
    root_mean_squared_error(y_val, pred) for pred in boost_model.staged_predict(X_val_t)
]
iterations = range(1, len(val_rmse_stages) + 1)
best_iter = int(np.argmin(val_rmse_stages)) + 1

plt.figure(figsize=(8, 5))
plt.plot(iterations, train_rmse_stages, label="Train RMSE")
plt.plot(iterations, val_rmse_stages, label="Validation RMSE")
plt.axvline(best_iter, color="grey", linestyle="--",
            label=f"Best validation iter = {best_iter}")
plt.xlabel("Number of boosting iterations (trees added)")
plt.ylabel("RMSE")
plt.title(f"Progressive boosting: {boost_name}")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Train RMSE falls from {train_rmse_stages[0]:.1f} to {train_rmse_stages[-1]:.1f} over "
      f"{len(train_rmse_stages)} iterations.")
print(f"Validation RMSE is lowest ({min(val_rmse_stages):.1f}) at iteration {best_iter}.")
print("If validation RMSE starts rising while train keeps falling, later trees are overfitting.")

## 11. Optional extension: XGBoost and LightGBM

> This section is an **optional extension**. Do the core Random Forest and Gradient Boosting work above first; only add these if time and interest allow.

`XGBoost` and `LightGBM` are widely used third-party gradient boosting implementations. They are usually faster and more configurable than classical Gradient Boosting, especially on larger tabular datasets — so they are worth knowing about as practical tools, even though they are not required for the core teaching goals.

They are **separate Python packages** (not part of scikit-learn), so they only run if the teaching environment has them installed. The cell below tries to import each one and skips it cleanly if it is missing, so the notebook always runs either way. If we decide to include this extension in the live session, `xgboost` and `lightgbm` can be added to the environment file.

In [ ]:
optional_models = {}

try:
    from xgboost import XGBRegressor

    optional_models["XGBoost"] = Pipeline(steps=[
        ("preprocess", make_preprocessor(X_train, scale_numeric=False)),
        ("model", XGBRegressor(
            objective="reg:squarederror",
            n_estimators=180,
            learning_rate=0.06,
            max_depth=4,
            min_child_weight=5,
            subsample=0.9,
            colsample_bytree=1.0,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=1,
            tree_method="hist",
        )),
    ])
except ImportError:
    print("XGBoost is not importable in the current notebook kernel. Skipping XGBoost.")

try:
    from lightgbm import LGBMRegressor

    optional_models["LightGBM"] = Pipeline(steps=[
        ("preprocess", make_preprocessor(X_train, scale_numeric=False)),
        ("model", LGBMRegressor(
            n_estimators=180,
            learning_rate=0.06,
            num_leaves=31,
            min_child_samples=25,
            subsample=0.9,
            colsample_bytree=1.0,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=1,
            verbosity=-1,
        )),
    ])
except ImportError:
    print("LightGBM is not importable in the current notebook kernel. Skipping LightGBM.")

optional_results = []

for model_name, model in optional_models.items():
    print(f"Fitting optional model: {model_name}")
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="X does not have valid feature names.*",
            category=UserWarning,
        )
        model.fit(X_train, y_train)
        fitted_models[model_name] = model
        results, predictions = evaluate_on_splits(model_name, model, include_train=True)
    optional_results.append(results)
    all_predictions[model_name] = predictions

if optional_results:
    comparison = pd.concat([comparison] + optional_results, ignore_index=True)
    display_metric_table(pd.concat(optional_results, ignore_index=True))
else:
    print("No optional third-party boosting models were added.")

print("\nModels currently available for downstream plots and diagnostics:")
for model_name in fitted_models:
    print("-", model_name)

## 12. Compare validation and test metrics

The **validation** set guides model choice; the **test** set is the final held-out check we only read *after* choosing.

**What to look for in the table and plot:**

- Pick the model with the best **validation** RMSE, then confirm it is also strong on **test**. Consistency between the two is a good sign.
- A model that looks great on validation but noticeably worse on test is a **red flag** — it may have been (indirectly) favoured by the validation split. The parallel Validation/Test lines in the plot make this easy to spot: watch for any model whose two markers jump apart.


In [ ]:
summary = comparison[comparison["Split"].isin(["Validation", "Test"])].copy()
summary = summary[["Model", "Split", "MAE", "RMSE", "R2"]]

display(summary.style.format({"MAE": "{:.2f}", "RMSE": "{:.2f}", "R2": "{:.3f}"}))

plt.figure(figsize=(8, 4.5))
for split_name, marker in [("Validation", "o"), ("Test", "s")]:
    subset = summary[summary["Split"] == split_name]
    plt.plot(subset["Model"], subset["RMSE"], marker=marker, label=split_name)

plt.ylabel("RMSE")
plt.title("Model comparison by split")
plt.xticks(rotation=20, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

## 13. Parity plots on the test set

Each panel plots **observed vs predicted** CHF for one model; perfect predictions lie on the diagonal one-to-one line.

**How to read them:** a tight cloud hugging the diagonal is best. Points **above** the line are under-predictions, **below** are over-predictions. Watch for **saturation** — a flattening at high CHF where a model stops climbing with the target, a sign it cannot reach extreme values. Comparing panels side by side shows which model has the tightest, least biased spread.


In [ ]:
test_predictions = {
    model_name: predictions["Test"]
    for model_name, predictions in all_predictions.items()
}

lims = [
    min(float(y_test.min()), *(float(np.min(pred)) for pred in test_predictions.values())),
    max(float(y_test.max()), *(float(np.max(pred)) for pred in test_predictions.values())),
]

plot_cols = 3
plot_rows = int(np.ceil(len(test_predictions) / plot_cols))
fig, axes = plt.subplots(plot_rows, plot_cols, figsize=(4.2 * plot_cols, 4.0 * plot_rows), sharex=True, sharey=True)
axes = axes.ravel()

for ax, (model_name, pred) in zip(axes, test_predictions.items()):
    ax.scatter(y_test, pred, alpha=0.55, s=18)
    ax.plot(lims, lims)
    ax.set_title(model_name)
    ax.set_xlabel("Observed CHF")
    ax.set_ylabel("Predicted CHF")

for ax in axes[len(test_predictions):]:
    ax.set_visible(False)

plt.suptitle("Parity plots - test set")
plt.tight_layout()
plt.show()

## 14. Residual plots on the test set

Residual = observed − predicted, plotted against the prediction. You want a **flat, structureless band centred on zero**.

**How to read them:** a **slope or curve** means systematic bias (the model is consistently off in some range); a **funnel** that widens means the error grows with CHF. A residual cloud drifting below zero at high predictions is the same saturation you may see as flattening in the parity plot — read the two plots together.


In [ ]:
plot_cols = 3
plot_rows = int(np.ceil(len(test_predictions) / plot_cols))
fig, axes = plt.subplots(plot_rows, plot_cols, figsize=(4.2 * plot_cols, 4.0 * plot_rows), sharex=False, sharey=True)
axes = axes.ravel()

for ax, (model_name, pred) in zip(axes, test_predictions.items()):
    residuals = y_test - pred
    ax.scatter(pred, residuals, alpha=0.55, s=18)
    ax.axhline(0.0)
    ax.set_title(model_name)
    ax.set_xlabel("Predicted CHF")
    ax.set_ylabel("Residual\nObserved - Predicted")

for ax in axes[len(test_predictions):]:
    ax.set_visible(False)

plt.suptitle("Residual plots - test set")
plt.tight_layout()
plt.show()

## 15. Regime-wise diagnostics

A model can look good **on average** while quietly failing in one corner of the operating space — a particular pressure, mass-flux, or quality band, or one data source. Averages hide this. Here we split the test set into quantile **bands** for each feature and recompute the error inside every band.

**What to look for:** scan each band's MAE / RMSE for a **spike** — a regime where the error is far above the model's overall average. That is where you would *not* trust the model in practice, even if its headline score is excellent. Also compare models *within* a band: the best-on-average model is not always the best in every regime.


In [ ]:
def prediction_frame(df, y_true, model_predictions):
    out = df[[id_col, reference_col] + feature_cols + [target_col]].copy()
    for model_name, pred in model_predictions.items():
        out[f"{model_name} prediction"] = pred
        out[f"{model_name} abs error"] = np.abs(y_true.to_numpy() - pred)
        out[f"{model_name} residual"] = y_true.to_numpy() - pred
    return out


def add_quantile_band(df, column, band_name, q=4):
    out = df.copy()
    try:
        out[band_name] = pd.qcut(out[column], q=q, duplicates="drop")
    except ValueError:
        out[band_name] = pd.cut(out[column], bins=q)
    return out


def regime_error_table(df, group_col, model_names):
    rows = []
    for group_value, group in df.groupby(group_col, observed=False):
        row = {"Regime": str(group_value), "Rows": len(group)}
        for model_name in model_names:
            row[f"{model_name} MAE"] = group[f"{model_name} abs error"].mean()
            row[f"{model_name} RMSE"] = root_mean_squared_error(
                group[target_col], group[f"{model_name} prediction"]
            )
        rows.append(row)
    return pd.DataFrame(rows)


test_diagnostics = prediction_frame(test_df, y_test, test_predictions)
display(test_diagnostics.head())

model_names = list(test_predictions.keys())

for column, band_name in [
    ("Pressure (kPa)", "Pressure band"),
    ("Mass Flux (kg/m^2/s)", "Mass flux band"),
    ("Outlet Quality", "Outlet quality band"),
]:
    if column in test_diagnostics.columns:
        test_diagnostics = add_quantile_band(test_diagnostics, column, band_name, q=4)
        print(f"\n{band_name} errors:")
        display(regime_error_table(test_diagnostics, band_name, model_names).style.format(precision=2))

print("\nSource-wise errors:")
display(regime_error_table(test_diagnostics, reference_col, model_names).style.format(precision=2))

## 16. Plot one regime-wise error summary

The default choice is the best validation-RMSE model from the models available in this run.

In [ ]:
chosen_model = (
    comparison[comparison["Split"] == "Validation"]
    .sort_values("RMSE")
    .iloc[0]["Model"]
)
chosen_regime = "Pressure band"

plot_table = regime_error_table(test_diagnostics, chosen_regime, [chosen_model])

plt.figure(figsize=(8, 4))
plt.bar(plot_table["Regime"], plot_table[f"{chosen_model} MAE"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("MAE")
plt.title(f"{chosen_model}: test MAE by {chosen_regime}")
plt.tight_layout()
plt.show()

## 17. Cautious interpretability with permutation importance

Permutation importance asks how much validation performance worsens when one feature is shuffled. It is model-specific and data-split-specific, so treat it as a diagnostic rather than a physical proof.

> Contrast with Part 1 (Decision Trees). There we used the single tree's **impurity-based** `feature_importances_`, computed from how much each feature reduced training impurity. Here we use **permutation** importance, measured on the held-out validation set. Impurity importance is cheap but biased toward high-cardinality / large-scale features and is a *training-set* quantity; permutation importance reflects genuine held-out predictive value. When two methods disagree, that disagreement is itself a useful warning, not a contradiction to resolve by picking one number.

In [ ]:
importance_model_name = (
    comparison[comparison["Split"] == "Validation"]
    .sort_values("RMSE")
    .iloc[0]["Model"]
)
importance_model = fitted_models[importance_model_name]

# Some third-party estimators warn when sklearn's permutation routine passes
# internally shuffled arrays without feature names. The predictions are valid.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="X does not have valid feature names.*",
        category=UserWarning,
    )
    perm = permutation_importance(
        importance_model,
        X_val,
        y_val,
        n_repeats=8,
        random_state=42,
        scoring="neg_root_mean_squared_error",
        n_jobs=1,
    )

importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance mean": perm.importances_mean,
    "Importance std": perm.importances_std,
}).sort_values("Importance mean", ascending=False)

display(importance_df.style.format({"Importance mean": "{:.3f}", "Importance std": "{:.3f}"}))

plt.figure(figsize=(7, 4))
plt.barh(importance_df["Feature"][::-1], importance_df["Importance mean"][::-1])
plt.xlabel("Increase in validation RMSE when shuffled")
plt.title(f"Permutation importance - {importance_model_name}")
plt.tight_layout()
plt.show()

## 18. Training-domain coverage check

Tree ensembles still need domain checks. Rows outside the training feature range deserve caution, even if the model returns a number.

In [ ]:
train_min = X_train.min()
train_max = X_train.max()

coverage_rows = []
for split_name, X in [("Validation", X_val), ("Test", X_test)]:
    outside = (X < train_min) | (X > train_max)
    coverage_rows.append({
        "Split": split_name,
        "Rows": len(X),
        "Rows outside train min-max on any feature": int(outside.any(axis=1).sum()),
        "Fraction outside": outside.any(axis=1).mean(),
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df.style.format({"Fraction outside": "{:.3f}"}))

## 19. Out-of-domain evaluation: regime shift vs source shift

The three OOD files are **not interchangeable**. Each is a deliberately different kind of external challenge set, and reading them correctly matters more than the raw numbers:

- **`chf_OOD_Kim2000` — regime shift (clearest case).** It sits in a much **lower-pressure, lower-mass-flux** region than the core training data (here roughly 70%+ of its rows fall *below* the trained pressure range). This is the textbook "asked to operate where the core data barely reaches" case, and the coverage check flags it loudly.
- **`chf_OOD_Peterlongo1966` — regime shift (subtle, joint case).** A shift toward **larger mass flow rates at a larger pipe diameter (0.0151 m)**. Each individual feature still sits *inside* the training min-max, so the marginal coverage check reports ~0% outside — yet this large-diameter + high-flow *combination* is barely represented in training, so the model is really being asked to extrapolate. Lesson: **min-max coverage is necessary, not sufficient** — it checks features one at a time and misses joint shifts. (With the default three features, tube diameter is not even an input, so its shift is doubly invisible; set `use_all_physical_features = True` to bring diameter in.)
- **`chf_OOD_Lee1966` — source shift.** A different external source with a narrower, controlled operating window. All features stay within the trained range, so this is **not** extrapolation; it tests whether the model transfers to a new data source even when the operating envelope overlaps.

So think of them as two categories: **regime shift (Kim, Peterlongo)** and **source shift (Lee)**. Compare each model's error across the three, and note that an ensemble's advantage on in-domain test data does **not** guarantee robustness under either kind of shift.

> Aside — `chf_ong.csv`: the data folder also holds the larger master reference dataset (24k+ points, with source provenance). It is meant for understanding the overall scale, heterogeneity, and provenance of the CHF data — **not** as a teaching split. These labs deliberately use the smaller, targeted train / val / test files instead.

In [ ]:
def evaluate_external_file(filename, models, feature_cols, target_col):
    path = data_dir / filename
    if not path.exists():
        print(f"Skipping missing file: {filename}")
        return None

    df = pd.read_csv(path)
    X = df[feature_cols].copy()
    y = df[target_col].copy()

    outside = (X < train_min) | (X > train_max)

    rows = []
    for model_name, model in models.items():
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message="X does not have valid feature names.*",
                category=UserWarning,
            )
            pred = model.predict(X)
        row = regression_metrics(y, pred)
        row["File"] = filename
        row["Model"] = model_name
        row["Rows"] = len(df)
        row["Fraction outside train min-max"] = outside.any(axis=1).mean()
        rows.append(row)

    return pd.DataFrame(rows)


ood_result_tables = []
for filename in ood_files:
    result = evaluate_external_file(filename, fitted_models, feature_cols, target_col)
    if result is not None:
        ood_result_tables.append(result)

if ood_result_tables:
    ood_results = pd.concat(ood_result_tables, ignore_index=True)
    ood_results = ood_results[[
        "File", "Model", "Rows", "Fraction outside train min-max", "MAE", "RMSE", "R2"
    ]]
    display(ood_results.style.format({
        "Fraction outside train min-max": "{:.3f}",
        "MAE": "{:.2f}",
        "RMSE": "{:.2f}",
        "R2": "{:.3f}",
    }))
else:
    print("No OOD result tables were produced.")

## 20. Short interpretation questions

Please answer these briefly in markdown below:

1. Which model has the best validation RMSE? Does it also look best on the test set?
2. Do Random Forest and Gradient Boosting clearly improve on the single Decision Tree?
3. Are there regimes or data sources where the best average model still struggles?
4. On the OOD files, does the model behave differently on the regime-shift sets (Kim, Peterlongo) versus the source-shift set (Lee)? Did the min-max coverage check warn you in every case?
5. If XGBoost or LightGBM ran in your environment, did either justify the extra dependency over the core Gradient Boosting?
6. Which features appear important, and how cautious should you be about that interpretation?
7. If you had to recommend one model for a preliminary engineering report, what limitation would you state first?

In [ ]:
# Write your brief interpretation here.
#
# 1. Recommended model:
# 2. Evidence:
# 3. Regime/source concern:
# 4. Interpretability caution:
# 5. Safe-use recommendation:
#

## 21. Final checklist

- You used the same data split as Day 1.
- You compared Ridge, Decision Tree, and the two core ensembles (Random Forest and Gradient Boosting).
- You watched Gradient Boosting improve iteration by iteration.
- You understood where XGBoost and LightGBM fit as an optional extension, even if the packages were not installed.
- You used validation metrics before looking at test conclusions.
- You inspected at least one regime-wise diagnostic.
- You distinguished regime-shift (Kim, Peterlongo) from source-shift (Lee) out-of-domain data.
- You wrote one caution for deployment or engineering use.

In [ ]:
print("Day 2 Lab 2 (Part 2) notebook completed successfully.")
print("You have compared baseline, bagging, boosting, and optional advanced boosting models for CHF prediction.")